# Visualization

This notebook serves to visualize the results of the models.

In [1]:
import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline
import importlib

sys.path.append("..")
sys.path.append("../code")
sys.path.append(os.path.join("..", 'models','Pointnet_Pointnet2_pytorch', 'models'))

from dataset import PCExtrusionSegmentationDataset
from models.DeepCAD.cadlib.visualize import vec2CADsolid
from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import create_CAD
from models.DeepCAD.cadlib.visualize import CADsolid2pc
from models.DeepCAD.utils.pc_utils import write_ply
import open3d as o3d
from metrics import ClassificationRunningScore
import torch

## Extrusion Segmentation

### Visualization

In [2]:
def get_trained_segmentation_pn2(model_path):
   
    model_name = 'pointnet2_sem_seg_msg'
    model = importlib.import_module(model_name)
    num_classes = 10
    classifier = model.get_model(num_classes)
    
    trained_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
    state_dict = trained_model['model_state_dict']
    classifier.load_state_dict(state_dict)
    config = trained_model['config']

    return classifier, config
    

In [3]:
import open3d as o3d
import matplotlib.pyplot as plt

def visualize_labeled_pc(points, labels):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    colors = plt.cm.tab10(labels / labels.max())[:, :3] 
    pcd.colors = o3d.utility.Vector3dVector(colors)

    o3d.visualization.draw_geometries([pcd])

In [4]:
def infer_segmentation_pn2(model, pc, show=True):
    pc = pc.unsqueeze(0)
    pc = pc.transpose(2, 1)
    pred_logits, _ = model(pc)
    pred_logits = pred_logits.data.view(-1, 10)
    pred = pred_logits.max(1)[1]
    return pred

In [12]:
run_name = "partseg_overfit_on_first_5"
model_path = os.path.join("..", "models", "trained_models", run_name, "ckpt_20.pth")
classifier, config = get_trained_segmentation_pn2(model_path)
config

{'learning_rate': 0.001,
 'batch_size': 5,
 'max_epochs': 20,
 'optimizer': 'Adam',
 'model_type': 'pointnet2_sem_seg_msg',
 'save_interval': 20,
 'early_stopping': 20,
 'start_time': '2025-07-15_16-55-55',
 'lr_type': 'step',
 'gpu': False,
 'final_epoch': 20,
 'training_time_min': 11.3}

In [13]:
train_dataset = PCExtrusionSegmentationDataset("../data", 'train', use_normals=False, verbose=False)
#val_dataset = PCExtrusionSegmentationDataset("../data", 'validation', use_normals=False, verbose=False)
#test_dataset = PCExtrusionSegmentationDataset("../data", 'test', use_normals=False, verbose=False)

In [7]:
index = 0
data = train_dataset[index]
pc = data['pc']
label = data['label']

In [116]:
visualize_labeled_pc(pc, label)
pred_labels = infer_segmentation_pn2(classifier, pc)
visualize_labeled_pc(pc, pred_labels)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [14]:
for i in range(0, 10):
    data = train_dataset[i]
    pc = data['pc']
    label = data['label']
    visualize_labeled_pc(pc, label)
    pred_labels = infer_segmentation_pn2(classifier, pc)
    visualize_labeled_pc(pc, pred_labels)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARN

In [9]:
def save_pc_with_labels_for_blender(points, labels, name):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.numpy())
    
    # Store labels as a scalar color channel (temporary workaround using red)
    colors = np.zeros((points.shape[0], 3))
    color_map = np.array([
        [0.894, 0.102, 0.110],  # class 0
        [0.216, 0.494, 0.722],  # class 1
        [0.302, 0.686, 0.290],  # class 2
        [0.596, 0.306, 0.639],  # class 3
        [1.000, 0.498, 0.0],    # class 4
        [1.000, 1.000, 0.2],    # class 5
        [0.651, 0.337, 0.157],  # class 6
        [0.969, 0.506, 0.749],  # class 7
        [0.6,   0.6,   0.6],    # class 8
        [0.1,   0.1,   0.1],    # class 9
    ])
    colors = color_map[labels.numpy()]
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    o3d.io.write_point_cloud(os.path.join("examples", name + ".ply"), pcd)

In [17]:
save_pc_with_labels_for_blender(pc, pred_labels, "pred_pc_2")

### Test Metrics: End-to-end pipeline

In [69]:
run_name = "complex_run"

test_metrics_1 = np.load(os.path.join("..", "models", "trained_models", run_name, "test_metrics.npz"), allow_pickle=True)
test_metrics_2 = pd.read_csv(os.path.join("..", "models", "trained_models", run_name, "test_metrics.csv"))
test_metrics_3 = np.load(os.path.join("..", "models", "trained_models", run_name, "test_cd.npy"), allow_pickle=True)

In [70]:
row = test_metrics_2.iloc[0]
max_key_len = max(len(k) for k in row.index)
for k, v in row.items():
    if k.endswith("loss"):
        print(f"{k:<{max_key_len}} : {v:.4f}")
    else:
        print(f"{k:<{max_key_len}} : {v * 100:.2f}%")

test_avg_cmd_acc  : 68.20%
test_avg_param_cc : 75.38%
test_cmd_loss     : 0.6139
test_param_loss   : 2.5389


In [71]:
ALL_COMMANDS = ['Line', 'Arc', 'Circle', 'EOS', 'SOL', 'Ext']
for cmd_acc, cmd in zip(list(test_metrics_1['epoch_per_cmd_acc_test'].squeeze()), ALL_COMMANDS):
    print(f"{cmd:<8} {cmd_acc * 100:.2f}%")

Line     80.04%
Arc      33.67%
Circle   59.13%
EOS      0.00%
SOL      64.07%
Ext      53.12%


In [72]:
ALL_ARGS = ["x", "y", "alpha", "f", "r", "theta", "phi", "gamma", "p_x", "p_y", "p_z", "s", "e_1", "e_2", "b", "u"]
for i, cmd in enumerate(test_metrics_1['epoch_per_param_acc_test'].squeeze()):
    print(ALL_COMMANDS[i])
    for j, param in enumerate(cmd):
        if param != 0:
            print(f"{ALL_ARGS[j]:<5}: {param * 100:.2f}%")
    print()

Line
x    : 66.23%
y    : 64.06%

Arc
x    : 62.45%
y    : 62.07%
alpha: 62.32%
f    : 85.60%

Circle
x    : 90.26%
y    : 89.19%
r    : 88.28%

EOS

SOL

Ext
theta: 93.36%
phi  : 93.02%
gamma: 93.30%
p_x  : 66.64%
p_y  : 75.33%
p_z  : 77.24%
s    : 62.07%
e_1  : 66.34%
e_2  : 99.15%
b    : 93.09%
u    : 96.57%



In [73]:
cd_list = list(test_metrics_3)
median_cd = np.nanmean(cd_list)
ir = np.isnan(cd_list).sum() / len(cd_list)
print(f"Median cd * 10^3: {median_cd*10e3:.2f}\nInvalid ratio   : {ir * 100:.2f}%")

Median cd * 10^3: 398.14
Invalid ratio   : 26.27%


In [22]:
import pickle
with open("../models/trained_models/complex_run/test_per_sl_metrics.pkl", "rb") as f:
    lol = pickle.load(f)

In [45]:
sum = 0
mean_list = []
for k in lol['cmd_acc'].keys():
    mean_list.append(np.nanmean(lol['param_acc'][k]))
sum

/var/folders/97/07fn6f7n48j71zk65dskp3gm0000gn/T/ipykernel_55205/1778500099.py:4: RuntimeWarning: Mean of empty slice
  mean_list.append(np.nanmean(lol['param_acc'][k]))


0

In [48]:
import pandas as pd
import torch

tensor = torch.randn(60, 17)
df = pd.DataFrame({
    "id": [1],
    "tensor": [tensor]
})

df


,id,tensor
0,1,"[[tensor(0.9530), tensor(-0.9114), tensor(-0.1..."


In [75]:
df = pd.read_pickle("../models/trained_models/complex_run/test_sample_results.pkl")

In [76]:
df

,id,set_id,cmd_acc,param_acc,cd,tgt_commands,pred_commands,tgt_args,pred_args,seq_len,pred_seq_len,cmd_count,cmd_correct,per_cmd_param_count,per_cmd_param_correct,param_count_total,param_correct_total,cmd_loss,param_loss,total_loss
0,00250456,0,0.999999,0.999995,0.004222,"[4, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 5]","[S, L, 223, 128, L, 223, 223, L, 128, 223, L, ...","[S, L, 223, 128, L, 223, 223, L, 128, 223, L, ...",6,6,"[4, 0, 0, 0, 1, 1]","[4, 0, 0, 0, 1, 1]","[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",19,19,0.027746,0.219510,0.247256
1,00440420,1,0.999999,0.555554,0.015578,"[4, 0, 0, 0, 0, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 0, 0, 0, 0, 5]","[S, L, 150, 106, L, 201, 106, L, 223, 128, L, ...","[S, L, 137, 100, L, 194, 99, L, 223, 128, L, 2...",10,10,"[8, 0, 0, 0, 1, 1]","[8, 0, 0, 0, 1, 1]","[[8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",27,15,0.208210,4.097722,4.305932
2,00819758,2,0.857142,0.642855,0.007716,"[4, 0, 0, 0, 0, 4, 2, 4, 2, 4, 2, 4, 2, 5]","[4, 0, 0, 0, 0, 0, 0, 4, 2, 4, 2, 4, 2, 5]","[S, L, 136, 108, L, 215, 108, L, 223, 128, L, ...","[S, L, 134, 117, L, 146, 117, L, 223, 128, L, ...",14,14,"[4, 0, 4, 0, 5, 1]","[4, 0, 3, 0, 4, 1]","[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",28,18,0.787318,4.146999,4.934318
3,00239323,3,0.899999,0.499997,NaN,"[4, 1, 0, 1, 0, 4, 2, 4, 2, 5]","[4, 1, 0, 1, 0, 4, 2, 4, 2, 4, 2, 5, 4, 2]","[S, A, 176, 128, 128, 1, L, 176, 199, A, 128, ...","[S, A, 179, 128, 128, 1, L, 179, 223, A, 128, ...",10,14,"[2, 2, 2, 0, 3, 1]","[2, 2, 2, 0, 3, 0]","[[2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",18,9,1.001187,2.646271,3.647458
4,00420729,4,0.999999,0.842101,0.004134,"[4, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 5]","[S, L, 223, 128, L, 223, 179, L, 128, 179, L, ...","[S, L, 223, 128, L, 223, 176, L, 128, 176, L, ...",6,6,"[4, 0, 0, 0, 1, 1]","[4, 0, 0, 0, 1, 1]","[[4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",19,16,0.033987,1.388123,1.422110
5,00646280,5,0.125000,0.999968,0.004138,"[4, 2, 5, 4, 2, 5, 4, 0, 0, 0, 1, 5, 4, 0, 1, ...","[4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4]","[S, C, 176, 128, 48, E, 192, 64, 192, 32, 128,...","[S, C, 176, 128, 48, S, C, 139, 128, 8, S, C, ...",24,11,"[9, 3, 2, 0, 5, 5]","[0, 0, 1, 0, 2, 0]","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",3,3,2.321687,5.446623,7.768310
6,00503948,6,0.285714,0.687496,NaN,"[4, 1, 0, 1, 0, 1, 0, 1, 0, 4, 2, 4, 2, 5, 4, ...","[4, 1, 0, 0, 0, 4, 0, 4, 2, 4, 2, 5, 2, 2]","[S, A, 130, 126, 64, 1, L, 137, 126, A, 140, 1...","[S, A, 138, 128, 64, 1, L, 138, 223, L, 128, 2...",28,14,"[8, 8, 4, 0, 6, 2]","[3, 1, 2, 0, 2, 0]","[[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",16,11,2.222811,3.988213,6.211024
7,00864479,7,0.833333,0.823525,0.088053,"[4, 0, 0, 0, 0, 5]","[4, 1, 0, 0, 0, 5, 4, 5, 0, 0, 0, 0, 0, 0, 0, ...","[S, L, 144, 70, L, 207, 70, L, 223, 128, L, 12...","[S, A, 135, 128, 64, 1, L, 223, 128, L, 223, 1...",6,22,"[4, 0, 0, 0, 1, 1]","[3, 0, 0, 0, 1, 1]","[[3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",17,14,1.118748,4.074796,5.193544
8,00150970,8,0.999999,0.999994,0.000265,"[4, 2, 4, 2, 5]","[4, 2, 4, 2, 5]","[S, C, 176, 128, 48, S, C, 176, 128, 34, E, 12...","[S, C, 176, 128, 48, S, C, 176, 128, 32, E, 12...",5,5,"[0, 0, 2, 0, 2, 1]","[0, 0, 2, 0, 2, 1]","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",17,17,0.073499,0.739965,0.813464
9,00503096,9,0.999999,0.842101,0.010122,"[4, 0, 0, 0, 0, 5]","[4, 0, 0, 0, 0, 5]","[S, L, 223, 128, L, 223, 198, L, 128, 198, L, ...","[S, L, 223, 128, L, 223, 176, L, 128, 176, L, ...",6,6,"[4, 0, 0, 0, 1, 1]","[4,

In [78]:
with pd.option_context('display.max_colwidth', None):
    print(df.iloc[4])

id                                                                                                                                                                                                                                                                                                                           00420729
set_id                                                                                                                                                                                                                                                                                                                              4
cmd_acc                                                                                                                                                                                                                                                                                                                      0.999999
param_acc             

In [46]:
mean_list

[nan,
 nan,
 nan,
 0.9610084934329325,
 0.4750346122339855,
 0.9416601521398759,
 0.8697880957181746,
 0.7230557099465955,
 0.7693458079636033,
 0.7263734852012654,
 0.7343701399919536,
 0.7418650961247188,
 0.6761182473198688,
 0.6282452779047242,
 0.599262784969775,
 0.6909895222594047,
 0.6147784785609313,
 0.6334841150601139,
 0.6621358121160776,
 0.602136050108936,
 0.5838010302676284,
 0.588676257050602,
 0.45031167512816384,
 0.5617781980721543,
 0.5860317584138345,
 0.614639762119564,
 0.508091321003549,
 0.5536952739024815,
 0.4978552930869444,
 0.5658944837641952,
 0.6410215659964001,
 0.5513169825479817,
 0.5085338670597125,
 0.471389985187659,
 0.664699871101094,
 0.4264855108269984,
 0.6051901910040456,
 0.4759348435071233,
 0.49815716682173317,
 0.38844903660585495,
 0.4478438506139724,
 0.5359788392552451,
 0.4928224720183001,
 0.5647445487575993,
 0.47872022756361526,
 0.4142363838452901,
 0.45463525265920857,
 0.5344879533404481,
 0.5442967007945384,
 0.339431015407179

In [41]:
lol['cmd_acc'][5]

[0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.7999990400011519,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.7999990400011519,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.9999988000014398,
 0.99999880000144,
 0.99999880000144,
 0.9999988000014398,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.9999988000014398,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.3999995200005759,
 0.7999990400011519,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144,
 0.99999880000144

### Test Metrics: Segmentation

In [13]:
run_name = "partseg_miou_run"

test_metrics_1 = np.load(os.path.join("..", "models", "trained_models", run_name, "test_metrics.npz"), allow_pickle=True)
test_metrics_2 = pd.read_csv(os.path.join("..", "models", "trained_models", run_name, "test_metrics.csv"))

In [14]:
test_metrics_1['class_iou'], test_metrics_1['class_acc']

(array([[0.88004574, 0.39655013, 0.23347675, 0.17249301, 0.14397624,
         0.10523883, 0.09056356, 0.0794565 , 0.11344418, 0.09527454]]),
 array([[0.95592844, 0.55201697, 0.36310666, 0.25237258, 0.22397793,
         0.15413569, 0.1289525 , 0.10086377, 0.1409144 , 0.11010391]]))

In [15]:
test_metrics_2

,mIoU,acc,mean_acc,loss
0,0.231052,0.821965,0.298237,0.696476


In [89]:
import random
random.sample([0,1,2,3,4,5,6,7,8,9],1)

[6]

In [107]:
random.randint(0, 9)

9